# Importación de librerías

In [1]:
from pathlib import Path
import os

import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image
from torchvision.transforms import v2

# Importación de dataset

Ponemos las carpetas base para entrada (`data/raw/images`) y para salida (`data/processed/images`) y recogemos todas las rutas hasta las imágenes.

In [2]:
DATA_ROOT = Path("E:/proyecto/data")
INPUT_DIR = DATA_ROOT / "raw" / "images"
OUTPUT_DIR = DATA_ROOT  / "processed" / "images"

Utilizando los parámetros "implícitos" de la ruta que habíamos creado, creamos un DataFrame con elementos con el siguiente formato:
*   `image_path`: ruta completa a la imagen.
*   `species`: nombre de la especie (con el formato utilizado para carpetas).
*   `plant_part`: parte de la planta (zona de imagen que se está mostrando).

In [3]:
rows = []

for image_path in sorted(INPUT_DIR.rglob("*.jpg")):
    relative = image_path.relative_to(INPUT_DIR)

    species = relative.parts[0]
    plant_part = relative.parts[1]

    rows.append(
        {
            "image_path": str(image_path),
            "species": species,
            "plant_part": plant_part
        }
    )

df = pd.DataFrame(rows)

## Visualización de dataset

Podemos ver la cantidad de imágenes que tenemos, la cantidad de especies (que se mantiene en 30) y la cantidad de 'partes' de plantas que se muestran en nuestra estructura.

In [4]:
print(f"Imágenes: {len(df):,}")
print(f"Especies endémicas: {df['species'].nunique():,}")
print(f"Partes de plantas: {df['plant_part'].nunique():,}")
display(df.head())

Imágenes: 2,355
Especies endémicas: 30
Partes de plantas: 6


,image_path,species,plant_part
0,E:\proyecto\data\raw\images\aeonium_arboreum\b...,aeonium_arboreum,bark
1,E:\proyecto\data\raw\images\aeonium_arboreum\b...,aeonium_arboreum,bark
2,E:\proyecto\data\raw\images\aeonium_arboreum\f...,aeonium_arboreum,flower
3,E:\proyecto\data\raw\images\aeonium_arboreum\f...,aeonium_arboreum,flower
4,E:\proyecto\data\raw\images\aeonium_arboreum\f...,aeonium_arboreum,flower


Otra cosa que podemos hacer es revisar la cantidad de imágenes por planta y/o parte de planta (ambos agrupados).

In [5]:
summary = (
    df.groupby(["species", "plant_part"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["species", "count"])
)

display(summary.head(10))
display(summary.groupby("species")["count"].sum().sort_values(ascending=False).head(10))

,species,plant_part,count
0,aeonium_arboreum,bark,2
2,aeonium_arboreum,habit,3
3,aeonium_arboreum,leaf,20
1,aeonium_arboreum,flower,29
4,aichryson_laxum,bark,1
6,aichryson_laxum,habit,3
5,aichryson_laxum,flower,17
7,aichryson_laxum,leaf,18
8,aizoon_canariense,bark,1
13,aizoon_canariense,other,1


species
rumex_lunaria                 240
lavandula_canariensis         167
canarina_canariensis          139
sonchus_canariensis           125
sonchus_acaulis               108
erysimum_scoparium            100
echium_wildpretii              98
pinus_canariensis              98
pterocephalus_lasiospermus     89
dracaena_draco                 78
Name: count, dtype: int64

# División de dataset en training / evaluation

Dividimos el dataset entre entrenamiento y validación (no lo llamamos testeo porque el entrenador llama a la variable "eval_dataset").

Nos debemos asegurar de estratificar los datos (`stratify`) para obtener los mejores resultados posibles.

In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["species"],
)

train_df = train_df.reset_index()
val_df = val_df.reset_index()

# Aplicación de data augmentation y exportación de datos

## Data Augmentation

Aplicamos data augmentation utilizando TorchVision, en específico, el paquete  `torchvision.transforms.v2`. Como clase base usamos `Compose`, una clase que permite encadenar varias transformaciones, de forma similar a como hacíamos con el `Sequential` de Keras. Con este, hacemos:
*   `RandomHorizontalFlip`: con una posibilidad del 50% (`p=0.5`), la imagen puede rotarse horizontalmente.
*   `RandomRotation`: establece una pequeña rotación a la imagen, de entre -15 y 15 grados (`degrees=15`).
*   `ColorJitter`: cambia aleatoriamente la iluminación (`brightness`), contraste (`contrast`), saturación (`saturation`) y tono (`hue`) de la imagen.
    *   En este caso, igualmente utilizamos transformaciones moderadamente. El objetivo es que se diferencie un poco en las imágenes muy similares para que el modelo aprenda mejor. Transformaciones muy fuertes podrían quitar información valiosa, principalmente sobre los colores, al VLM, así que no debemos ser cuidadosos.
*   `RandomApply(GaussianBlur)`: aplica un desenfoque gaussiano con un 20% de posibilidades. Esto obliga al VLM a aprender patrones generales, en lugar de fijarse demasiado en los detalles.
*    `RandomPerspective`: introduce una pequeña distorsión de la perspectiva, con un 20% de probabilidad. Esto ayuda a ajustarse a cambios en el punto de vista.

In [10]:
transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),

    v2.RandomRotation(degrees=15),

    v2.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.2,
        hue=0.05,
    ),

    v2.RandomApply([
        v2.GaussianBlur(kernel_size=3)
    ], p=0.2),

    v2.RandomPerspective(
        distortion_scale=0.15,
        p=0.2,
    ),
])

images = []
for index, row in train_df.iterrows():
    image_path = row['image_path']
    image = Image.open(image_path)

    images.append(transform(image)) # Añadimos a una lista todas las imágenes aumentdas.

## Exportación de datos

Y finalmente, guardamos las nuevas imágenes, tanto las aumentadas de entrenamiento como las de validación y aprovechamos para dividirlas en carpetas distintas.

In [11]:
for index, row in train_df.iterrows():
    image = images[index]
    species = row['species']
    plant_part = row['plant_part']

    new_path = OUTPUT_DIR / "train" / species / plant_part / f"{index}.jpg" # Les volvemos a asignar nombres triviales.
    os.makedirs(new_path.parent, exist_ok=True)

    with open(new_path, 'wb') as f:
        image.save(f)

for index, row in val_df.iterrows():
    image = Image.open(row['image_path'])
    species = row['species']
    plant_part = row['plant_part']

    new_path = OUTPUT_DIR / "val" / species / plant_part / f"{index}.jpg" # Les volvemos a asignar nombres triviales.
    os.makedirs(new_path.parent, exist_ok=True)

    with open(new_path, 'wb') as f:
        image.save(f)